# 12.2: Computing perplexity in practice

With the Hugging Face `transformers` library, computing perplexity takes a few lines. The model returns the mean cross-entropy loss when you pass the input IDs as labels; the library shifts the labels internally so that each position predicts the next token.


In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def perplexity(text: str, model_name: str = "gpt2", window: int = 512, stride: int = 256) -> float:
    """Perplexity of `text` using a sliding window so long texts fit in context."""
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).eval()
    ids = tok(text, return_tensors="pt").input_ids[0]

    total_nll, total_tokens, prev_end = 0.0, 0, 0
    for start in range(0, len(ids), stride):
        end = min(start + window, len(ids))
        chunk = ids[start:end].unsqueeze(0)
        labels = chunk.clone()
        # Only score tokens not already scored by the previous window.
        n_new = end - prev_end
        labels[:, :-n_new] = -100          # -100 = ignore in the loss
        with torch.no_grad():
            loss = model(chunk, labels=labels).loss
        scored = (labels[:, 1:] != -100).sum().item()   # first position is never scored
        total_nll += loss.item() * scored
        total_tokens += scored
        prev_end = end
        if end == len(ids):
            break
    return math.exp(total_nll / total_tokens)
